In [20]:
# Walk up to repo root (identified by CLAUDE.md)
.root <- getwd()
while (!file.exists(file.path(.root, 'CLAUDE.md')) && dirname(.root) != .root)
  .root <- dirname(.root)
setwd(.root)
rm(.root)

suppressPackageStartupMessages({
  library(tidyverse)
  library(gt)
  library(ggthemes)
  library(scales)
  library(patchwork)
})

BASE_THEME <- theme_fivethirtyeight() +
  theme(
    axis.title        = element_text(size = 14),
    axis.text         = element_text(size = 13),
    panel.background  = element_rect(fill = 'white', color = NA),
    plot.background   = element_rect(fill = 'white', color = NA),
    legend.background = element_rect(fill = 'white', color = NA),
    legend.text       = element_text(size = 12),
    legend.title      = element_text(size = 13),
    strip.text        = element_text(size = 14, face = 'bold'),
    plot.title        = element_text(size = 20, face = 'bold'),
    plot.subtitle     = element_text(size = 13)
  )
theme_set(BASE_THEME)

dir.create('results/tables', recursive = TRUE, showWarnings = FALSE)

PARAM_ORDER <- c(
  'Intercept',
  'H. Displacement',
  'V. Displacement',
  'Proj. plate_x',
  'Proj. plate_z',
  'Release Speed',
  'Balls',
  'Strikes',
  'Timing Offset'
)

RESP_ORDER <- c('VAA', 'HAA', 'Tilt')

TREATMENT_ROWS <- c('H. Displacement', 'V. Displacement')

# Mediator Models

In [21]:
med <- read_csv('results/tables/mediator_coefficients.csv', show_col_types = FALSE) |>
  mutate(
    param_label = factor(param_label, levels = PARAM_ORDER),
    resp_label  = factor(resp_label,  levels = RESP_ORDER)
  ) |>
  filter(!is.na(param_label)) |>
  arrange(param_label, resp_label)

med_wide <- med |>
  pivot_wider(
    id_cols     = param_label,
    names_from  = resp_label,
    values_from = c(coef, lower, upper),
    names_glue  = '{resp_label}_{.value}'
  )

resp_cols <- function(resp) {
  c(paste0(resp, '_coef'), paste0(resp, '_lower'), paste0(resp, '_upper'))
}

all_data_cols <- unlist(lapply(RESP_ORDER, resp_cols))

tbl_med <- med_wide |>
  select(param_label, all_of(all_data_cols)) |>
  gt(rowname_col = 'param_label') |>
  tab_stubhead(label = html('<b>Parameter</b>')) |>
  tab_spanner(label = 'VAA (Deg)',  columns = resp_cols('VAA'),  id = 'sp_vaa')  |>
  tab_spanner(label = 'HAA (Deg)',  columns = resp_cols('HAA'),  id = 'sp_haa')  |>
  tab_spanner(label = 'Tilt (Deg)', columns = resp_cols('Tilt'), id = 'sp_tilt') |>
  cols_label(
    VAA_coef   = html('<b>Coef</b>'),  VAA_lower  = html('<b>Lower</b>'),  VAA_upper  = html('<b>Upper</b>'),
    HAA_coef   = html('<b>Coef</b>'),  HAA_lower  = html('<b>Lower</b>'),  HAA_upper  = html('<b>Upper</b>'),
    Tilt_coef  = html('<b>Coef</b>'),  Tilt_lower = html('<b>Lower</b>'),  Tilt_upper = html('<b>Upper</b>')
  ) |>
  fmt_number(columns = everything(), decimals = 3) |>
  # left border between metric groups
  tab_style(
    style     = cell_borders(sides = 'left', color = '#aaaaaa', weight = px(1)),
    locations = list(
      cells_body(columns = c(resp_cols('VAA')[1], resp_cols('HAA')[1], resp_cols('Tilt')[1])),
      cells_column_labels(columns = c(resp_cols('VAA')[1], resp_cols('HAA')[1], resp_cols('Tilt')[1]))
    )
  ) |>
  # highlight treatment rows
  tab_style(
    style     = list(cell_fill(color = '#eef2ff'), cell_text(weight = 'bold')),
    locations = cells_stub(rows = param_label %in% TREATMENT_ROWS)
  ) |>
  tab_style(
    style     = cell_fill(color = '#eef2ff'),
    locations = cells_body(rows = param_label %in% TREATMENT_ROWS)
  ) |>
  # color positive/negative coef values
  tab_style(
    style     = cell_text(color = '#27ae60', weight = 'bold'),
    locations = cells_body(
      columns = c(VAA_coef, HAA_coef, Tilt_coef),
      rows    = VAA_coef > 0
    )
  ) |>
  tab_style(
    style     = cell_text(color = '#c0392b', weight = 'bold'),
    locations = cells_body(
      columns = c(VAA_coef, HAA_coef, Tilt_coef),
      rows    = VAA_coef < 0
    )
  ) |>
  tab_style(
    style     = cell_text(style = 'italic'),
    locations = cells_stub(rows = !(param_label %in% TREATMENT_ROWS))
  ) |>
  tab_style(
    style     = cell_text(weight = 'bold'),
    locations = cells_column_spanners()
  ) |>
  tab_source_note(
    source_note = 'Coef = mixed-effects OLS estimate; Lower/Upper = 95% CI. Blue rows = treatment (post-commit displacement).'
  ) |>
  tab_options(
    table.font.size                   = px(13),
    column_labels.font.size           = px(12),
    table.border.top.style            = 'solid',
    table.border.top.width            = px(2),
    table.border.top.color            = 'black',
    table_body.border.bottom.style    = 'solid',
    table_body.border.bottom.width    = px(2),
    table_body.border.bottom.color    = 'black',
    column_labels.border.top.style    = 'none',
    column_labels.border.bottom.style = 'solid',
    column_labels.border.bottom.width = px(1),
    column_labels.border.bottom.color = 'black',
    table.background.color            = 'white',
    stub.border.style                 = 'solid',
    stub.border.width                 = px(1),
    stub.border.color                 = '#aaaaaa'
  )

IRdisplay::display_html(as_raw_html(tbl_med))
tryCatch(
  gtsave(tbl_med, 'results/tables/mediator_coefficients.html'),
  error = function(e) message('gt HTML save skipped: ', conditionMessage(e))
)

Coef = mixed-effects OLS estimate; Lower/Upper = 95% CI. Blue rows = treatment (post-commit displacement).


## Outcome Model Validation

In [ ]:
dev_grad <- read_csv('data/viz_cache/cm_dev_gradient.csv',  show_col_types = FALSE)
ev_la_srf <- read_csv('data/viz_cache/cm_ev_la_surface.csv', show_col_types = FALSE)
11
cat(sprintf(
  'Loaded | gradient bins: %d | EV/LA grid: %d points\n',
  nrow(dev_grad), nrow(ev_la_srf)
))


ERROR: Error: 'data/viz_cache/cm_dev_gradient.csv' does not exist in current working directory ('/Users/theoauyeung/Documents/pitcher_bat_path_exploration').


## Swing Deviation Gradient: Larger Deviations Hurt Outcomes

In [ ]:
# Reshape for a 3-panel strip: P(BIP), mean EV, mean delta run exp
grad_long <- dev_grad |>
  select(dev_mid, p_bip, pred_bip, mean_ev, mean_delta_rv) |>
  pivot_longer(c(p_bip, mean_ev, mean_delta_rv),
               names_to = 'outcome', values_to = 'value') |>
  mutate(outcome = recode(outcome,
    'p_bip'         = 'P(Ball in play)',
    'mean_ev'       = 'Mean exit velocity (mph)',
    'mean_delta_rv' = 'Mean run value (actual)'
  ),
  outcome = factor(outcome, levels = c('P(Ball in play)',
                                       'Mean exit velocity (mph)',
                                       'Mean run value (actual)')))

pred_bip_df <- dev_grad |>
  select(dev_mid, pred_bip) |>
  mutate(outcome = 'P(Ball in play)')

p_grad <- ggplot(grad_long, aes(x = dev_mid, y = value)) +
  geom_line(color = '#2980b9', linewidth = 1.3) +
  geom_point(color = '#2980b9', size = 3) +
  geom_line(
    data    = pred_bip_df |> filter(outcome == 'P(Ball in play)'),
    mapping = aes(x = dev_mid, y = pred_bip),
    color   = '#e74c3c', linewidth = 1.0, linetype = 'dashed'
  ) +
  facet_wrap(~ outcome, scales = 'free_y', ncol = 3) +
  labs(
    title    = 'Larger Swing Deviations Consistently Hurt Outcomes',
    subtitle = paste0(
      'Swings binned by absolute VAA deviation (realized vs intended), 10 equal-frequency bins\n',
      'Blue = actual outcomes  |  Red dashed = model-predicted P(BIP)  |  ',
      formatC(sum(dev_grad$n), format = 'd', big.mark = ','), ' competitive swings'
    ),
    x = 'Absolute VAA deviation from intended (deg)',
    y = NULL
  ) +
  theme(strip.text = element_text(size = 13, face = 'bold'))

options(repr.plot.width = 15, repr.plot.height = 6)
p_grad
ggsave('results/plots/diagnostics/cm_deviation_gradient.png', p_grad,
       width = 15, height = 6, dpi = 150)


## EV/LA → Run Value Surface

In [ ]:
# Overlay actual BIP outcomes as points (sample for readability)
# Load actual BIP data for overlay -- just use xrv_causal sample
set.seed(42)

p_surf <- ggplot(ev_la_srf,
                 aes(x = launch_speed, y = launch_angle, fill = pred_rv)) +
  geom_tile() +
  scale_fill_gradient2(
    low      = '#d73027',
    mid      = '#ffffbf',
    high     = '#1a9850',
    midpoint = 0,
    name     = 'Predicted\nrun value',
    labels   = function(x) sprintf('%.2f', x)
  ) +
  geom_hline(yintercept = c(10, 30), linetype = 'dashed',
             color = 'white', linewidth = 0.6, alpha = 0.7) +
  geom_vline(xintercept = 90, linetype = 'dashed',
             color = 'white', linewidth = 0.6, alpha = 0.7) +
  annotate('text', x = 108, y = 20, label = 'Sweet spot',
           color = 'white', size = 4.5, fontface = 'bold') +
  annotate('text', x = 65,  y = -35, label = 'Grounders',
           color = 'white', size = 3.5, alpha = 0.9) +
  annotate('text', x = 65,  y = 42, label = 'Pop-ups',
           color = 'white', size = 3.5, alpha = 0.9) +
  scale_x_continuous(expand = c(0, 0)) +
  scale_y_continuous(expand = c(0, 0)) +
  labs(
    title    = 'EV/LA Run Value Surface: Model Recovers Baseball Sweet Spot',
    subtitle = 'Predicted run value from XGBoost EV/LA model  |  White dashed = sweet-spot boundaries (90+ mph, 10-30 deg)',
    x = 'Exit velocity (mph)',
    y = 'Launch angle (deg)'
  )

options(repr.plot.width = 12, repr.plot.height = 7)
p_surf
ggsave('results/plots/diagnostics/cm_ev_la_surface.png', p_surf,
       width = 12, height = 7, dpi = 150)
